# CaPy v2 — Training

Tri-modal contrastive learning for drug discovery.

**Phase 1 Gate:** bi-modal mol↔morph compound-level R@10 > 10% AND alignment < 1.5

**Runtime:** GPU (H100 recommended, T4/V100 also work)

In [ ]:
# Clone repo and install (idempotent — safe to re-run)
import os
if not os.path.exists("/content/CaPy-v2"):
    !git clone https://github.com/ogngnaoh/CaPy-v2.git
else:
    %cd /content/CaPy-v2
    !git pull
    %cd /content

%cd /content/CaPy-v2
!pip install -e ".[dev]" -q

# Verify install
import torch
print(f"PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Download data (morphology, expression, metadata)
!python3 scripts/download.py

In [ ]:
# Preprocess: QC, normalize, scaffold split
!python3 scripts/preprocess.py

In [ ]:
# Quick sanity check — verify processed data
import json
from pathlib import Path

import pandas as pd

processed = Path("data/processed")
for split in ["train", "val", "test"]:
    df = pd.read_parquet(processed / f"{split}.parquet")
    print(f"{split}: {len(df)} treatments, {df.shape[1]} columns")

with open(processed / "feature_columns.json") as f:
    cols = json.load(f)
print(f"\nMorph features: {len(cols['morph_features'])}")
print(f"Expr features: {len(cols['expr_features'])}")

In [ ]:
# Optional: login to W&B for experiment tracking
# Skip this cell if you don't want W&B logging
import wandb
wandb.login()

## Phase 1: Bi-Modal Mol↔Morph Baseline

**Gate criteria:** compound-level R@10 > 10% AND alignment < 1.5

Run single seed first to verify, then multi-seed for robustness.

In [ ]:
# Phase 1 gate: bi-modal mol-morph (single seed)
!python3 scripts/train.py model=bi_mol_morph seed=42

## Multi-Seed Validation

Run 3 additional seeds to confirm Phase 1 gate result is robust.
All seeds should pass: compound R@10 > 10% AND alignment < 1.5.

In [ ]:
# Multi-seed validation (seeds 123, 456, 789)
for seed in [123, 456, 789]:
    print(f"\n{'='*60}")
    print(f"Training seed={seed}")
    print(f"{'='*60}")
    !python3 scripts/train.py model=bi_mol_morph seed={seed}

In [ ]:
# Compare multi-seed results
import torch

print(f"{'Seed':<8} {'Epoch':<8} {'compound R@10':<16} {'Alignment':<12} {'Uniform mol':<14} {'Uniform morph':<14} {'Gate'}")
print("-" * 90)

for seed in [42, 123, 456, 789]:
    path = f"checkpoints/bi_mol_morph_seed{seed}.pt"
    try:
        ckpt = torch.load(path, weights_only=False, map_location="cpu")
        metrics = ckpt.get("metrics", {})
        r10 = ckpt.get("best_metric", metrics.get("compound/mean_R@10", -1))
        align = metrics.get("align_mol_morph", -1)
        u_mol = metrics.get("uniform_mol", -1)
        u_morph = metrics.get("uniform_morph", -1)
        epoch = ckpt.get("epoch", -1)
        gate = "PASS" if (r10 > 0.10 and align < 1.5) else "FAIL"
        print(f"{seed:<8} {epoch:<8} {r10:<16.4f} {align:<12.4f} {u_mol:<14.4f} {u_morph:<14.4f} {gate}")
    except FileNotFoundError:
        print(f"{seed:<8} not found (not trained yet)")

print("\nGate: compound R@10 > 10% AND alignment < 1.5")

## Phase 2: Additional Bi-Modal Baselines + Tri-Modal

After multi-seed validation passes, train remaining baselines and tri-modal.

In [ ]:
# B5: Mol↔Expr bi-modal
!python3 scripts/train.py model=bi_mol_expr seed=42

In [ ]:
# B6: Morph↔Expr bi-modal
!python3 scripts/train.py model=bi_morph_expr seed=42

In [ ]:
# T1: Tri-modal (all 3 modalities)
!python3 scripts/train.py model=tri_modal seed=42

In [ ]:
# Compare all configs (single seed)
import torch

configs = [
    ("bi_mol_morph", 42),
    ("bi_mol_expr", 42),
    ("bi_morph_expr", 42),
    ("tri_modal", 42),
]

print(f"{'Config':<20} {'Epoch':<8} {'Best R@10':<12}")
print("-" * 42)

for config, seed in configs:
    path = f"checkpoints/{config}_seed{seed}.pt"
    try:
        ckpt = torch.load(path, weights_only=False, map_location="cpu")
        r10 = ckpt.get("best_metric", -1)
        epoch = ckpt.get("epoch", -1)
        print(f"{config:<20} {epoch:<8} {r10:<12.4f}")
    except FileNotFoundError:
        print(f"{config:<20} not found")

## Full Ablation Matrix (5 seeds)

Run the complete 20-run matrix (4 trained configs x 5 seeds) for statistical analysis.

In [ ]:
# Full ablation matrix with resume support
# --resume skips configs that already have checkpoints
!python3 scripts/run_ablations.py --matrix core --resume --configs B4 B5 B6 T1